# Experiment 8 - Hindi + Hinglish SFT on the merged v8 corpus

Chained: CPT on Hindi (`cpt-hindi`) -> SFT on `train_v8.jsonl`. One Kaggle session,
resumable via `wip/s4-hinglish-v8` every 900 s.

**Both T4s train.** One process per GPU under `torch.distributed.run`; rank 0 owns
the Hub. The old one-card run was not a missing flag - the Unsloth multi-GPU raise
is dead code in the installed wheel. See the DDP cell for the citation.

**What is different from v7.** v7 was 57,687 rows of one register: response words
p10/med/p90 = 40/47/52, ZERO rows under 15 words, ZERO chat rows, one topic field
value. So it answered "kaise ho bhai" with a 47-word paragraph. v8 merges three
pools, all re-gated locally (teach/build_v8_release.py, AUDIT PASS + masking PASS):
  61,132 teacher rows (judge>=4, 50 topics, 22% <=22 words, romanised Hinglish)
   2,575 human multi-turn conversations from oasst1_hi (median 4 turns)
   2,029 human WikiHow-Hindi how-to answers, 64% carrying English glosses
= 65,736 rows / 24.9M tokens, balanced in TOKENS not rows (long rows are 3.8x the
median and would otherwise dominate every batch: v6's verbose-collapse failure).

**Batches are grouped by length.** The collator pads each micro-batch to its
longest row and the trainer does not pack, so a shuffled file spent 36% of every
epoch on pad tokens. Same rows, same hyper-parameters, ~1/3 less compute.

**Do not run the data cell past its assertion.** Rebuild with teach/build_v8_release.py.

In [ ]:
import os
# Must precede every FastLanguageModel.from_pretrained call, CPT and SFT alike.
os.environ["UNSLOTH_AUTO_DEVICE_MAP"] = "0"   # 0 = no cross-card split

# ---------------------------------------------------------------- run config
# One repository, branches instead of a pile of repos.
#
# Why branches and not one linear history with different names: TimeCheckpoint
# pushes every 300 s because Kaggle local storage is not durable, so a 12 h
# session is ~144 commits on its own. Putting those in one shared history makes
# the repo enormous for no benefit. A branch is a clean pointer - `revision=`
# pins it, every commit of that line of work stays together, and `main` is the
# one thing you deploy.
HF_USER = "kumarsujitroy"
REPO    = f"{HF_USER}/LFM2.5-1.2B-Instruct-HI-Uncensored"

# The line of work. This is the only thing you edit.
BRANCH = "s4-hinglish-v8"

BASE_BRANCH = "cpt-hindi"

MAX_STEPS     = 4025   # exactly 1.0 epoch of v8: 65,721 rows x 0.98 / 16 per step.\n#                  # v6 ran 1.21 epochs and v7 ran 0.51 of one; FINDINGS 5.1 calls\n#                  # matching the flag without the budget 'under-training'. The eval\n#                  # curve still decides - cell 23 prints the step of minimum eval loss.
#                  # v6 ran 1500 on 19,775 = 1.21 epochs; 1500 here = 0.42.
#                  # Stop at the eval-loss minimum the run prints.
LEARNING_RATE  = 2e-4           # vendor SFT value, was 1.5e-4
CHECKPOINT_EVERY_S = 900
WIP           = f"wip/{BRANCH}"  # 5-minute checkpoints land here, never on a release

def _has_safetensors(repo, rev):
    try:
        from huggingface_hub import list_repo_files
        return any(f.endswith(".safetensors") for f in list_repo_files(repo, revision=rev))
    except Exception:
        return False

_base_ok = _has_safetensors(REPO, BASE_BRANCH)
if _base_ok:
    # snapshot_download returns a local dir that holds BOTH model.safetensors and
    # adapter_config.json; the load cell merges that adapter instead of double-adding.
    from huggingface_hub import snapshot_download
    BASE_REPO = snapshot_download(REPO, revision=BASE_BRANCH)
    BASE_REV  = BASE_BRANCH
else:
    # fresh repo, or the branch has not been pushed yet: fall back to the
    # upstream abliterated Instruct, which is what the chain started from.
    BASE_REPO = "zaakirio/LFM2.5-1.2B-Instruct-Uncensored"
    BASE_REV  = None
    print(f"  note: branch '{BASE_BRANCH}' has no weights yet, starting from upstream")

# the teacher snapshot this run trains on
DATA_FILE = "train_v8.jsonl"

print(f"repo    {REPO}")
print(f"  base  {BASE_REPO}" + (f" @ {BASE_REV}" if BASE_REV else ""))
print(f"  out   {REPO} @ {BRANCH}")
print(f"  wip   {REPO} @ {WIP}   (5-min checkpoints)")
print(f"  data  {DATA_FILE}")
print(f"  {MAX_STEPS} steps @ lr {LEARNING_RATE}")

# --- GPU guard. The accelerator is a Kaggle UI setting, not something the
# notebook controls: importing an .ipynb resets Settings -> Accelerator to
# "None" often enough that a silent CPU run is a real risk. Training on CPU
# would not error, it would just take days, so assert instead of assuming.
import torch as _torch
_gpus = _torch.cuda.device_count() if _torch.cuda.is_available() else 0
if _gpus == 0:
    raise RuntimeError(
        "No GPU visible. Settings (right sidebar) -> Accelerator -> "
        "GPU T4 x2, then RESTART the session. Nothing below will run on CPU."
    )
print(f"GPUs visible: {_gpus}  " + "  ".join(
    f"{_torch.cuda.get_device_name(i)}" for i in range(_gpus)))

## 0. Login

`push_to_hub` is the whole point: it is the only thing that survives a Kaggle VM
reset. Without a token every checkpoint push fails and the run is lost.

**One-time setup:** right sidebar -> **Add-Ons** -> **Secrets** -> **+ New Secret**
- Label `HF_USER`'s HF token, i.e. `hf_...`, with **`write`** scope
- **tick Notebook Access**
- **restart the session** - Kaggle injects secrets only at session start

The notebook is public, so there is deliberately no paste slot anywhere in it.

In [ ]:
import os
from huggingface_hub import login, whoami     # whoami is a function, not a CLI command

TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or ""
except Exception:
    pass
TOKEN = (TOKEN or os.environ.get("HF_TOKEN", "")).strip()

if not TOKEN:
    raise RuntimeError(
        "No HF token. Add-Ons (right sidebar) -> Secrets -> + New Secret\n"
        "  Label: HF_TOKEN   Value: your hf_ write token\n"
        "Then RESTART the session.")
if not TOKEN.startswith("hf_"):
    raise ValueError(f"Token does not look like an HF token: {TOKEN[:6]}...")

login(token=TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = TOKEN
print("logged in as:", whoami(token=TOKEN).get("name"))

In [ ]:
# unsloth MUST be imported before transformers anywhere in this notebook, or it
# warns and skips its patches. Every later `from transformers import ...` is
# then safe.
!pip install -q unsloth
!pip install -q "transformers==4.57.3"          # 4.57 is what unsloth is tested against
!pip install -q --no-deps trl==0.22.2
!pip install -q datasets==4.3.0 accelerate peft bitsandbytes hf_transfer

import unsloth
from unsloth import FastLanguageModel
print("unsloth imported first - OK")

## Crash-proof checkpointing

**A local save on Kaggle is not a backup.** `/kaggle/working` is destroyed when the
VM resets, taking every `save_steps` checkpoint with it. The only durable store is
the Hub, so this pushes the adapter on a wall-clock timer rather than a step count.
If the session dies at minute 38 you resume from minute 35.

**Watch for these lines.** If they never appear, the push is failing silently and a
reset destroys the run.

In [ ]:
import time
from transformers import TrainerCallback


class TimeCheckpoint(TrainerCallback):
    """Push the adapter + tokenizer to the Hub every `every_s` wall-clock seconds."""

    def __init__(self, model, repo, every_s=300, tokenizer=None, revision=None):
        self.model, self.repo, self.every = model, repo, every_s
        self.rev = revision          # WIP branch, so a 12 h session's ~144 five-minute
                                     # commits never land on a release branch
        self.tok, self.n, self.last = tokenizer, 0, time.time()

    def on_step_end(self, args, state, control, **kw):
        if time.time() - self.last >= self.every:
            self.n += 1
            tag = f"ckpt-s{state.global_step // 100 * 100:05d}"
            self.model.push_to_hub(self.repo, save_name=tag, revision=self.rev, private=True)
            if self.tok is not None:
                # both, or the checkpoint is only half-recoverable
                self.tok.push_to_hub(self.repo, save_name=tag, revision=self.rev, private=True)
            print(f"[{time.strftime('%H:%M:%S')}] pushed {tag} @ step {state.global_step}",
                  flush=True)
            self.last = time.time()
        return control

## 1. Build the training data

Self-contained, runs here on Kaggle. Writes all three session files into
`/kaggle/working/`. Set BUILD_DATA = False to skip and use the repo's copies.

Sources, one per bucket because no single dataset is good at all three:

| session | source | why |
|---|---|---|
| s1 | Sujalvc AAG_anchored | conversational backbone, 68-word replies |
| s1 | arena | real humans typing; keep only the 30% over 20 words |
| s1 | COMI-LINGUA RH | 3-expert-annotated romanized (EMNLP 2025) |
| s2 | smangrul Devanagari + zicsx | conversational + clean pool |
| s2 | COMI-LINGUA DH | 3-expert-annotated Devanagari |
| s3 | dolly-15k | human-written English anchor |

### Every row faces the same 20-word floor

There is no per-source exemption. That is the whole point, and it was the bug:
the build used to admit casual at 3 words and arena at 6 so the row counts would
come out right. `casual` is 1500 rows of which **0 survive 20 words** - 92% of its
replies are under 10 and none exceed 40. The model reproduced that faithfully for
690 steps, which is the 4-word collapse.

At 20 words the s1 median reply goes **8 -> 40**. The teacher writes 37-49, so
training data and teacher end up on the same distribution instead of two apart.

**Read the median-reply-words line before training.** Under 20 means fragments, not
conversation. Fewer real rows beat padded bad ones - if a session comes out thin,
that is the true count and the teacher generation is the volume answer.

In [ ]:
# Self-contained data build on Kaggle.
# The teacher snapshot lives in the PRIVATE dataset repo as
# train_v8.jsonl; cell 9 fetches it. Rebuilding the oldes it. Rebuilding the old s1/s2 mixtures here would overwrite it.
BUILD_DATA = False
# Every source that measured well gets used. Source count is not a constraint -
# each session trains independently and the downloads stream.
#   s1  Sujalvc + arena + casual + COMI-LINGUA   (romanized Hinglish)
#   s2  smangrul + zicsx + COMI-LINGUA           (Devanagari Hindi)
#   s3  smangrul English-bearing rows
HINDI_TARGET    = 6000
HINGLISH_TARGET = 8500
ENGLISH_TARGET  = 4000

if not BUILD_DATA:
    print("BUILD_DATA=False - the next cell will use the repo's jsonl")
else:
    import csv, json, os, random, re, urllib.request

    def grab(url, dest):
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if not os.path.exists(dest):
            with urllib.request.urlopen(url, timeout=900) as r, open(dest, "wb") as f:
                f.write(r.read())
        return dest

    def stream_csv(url, dest, i_c, o_c, cap):
        """Arena/casual are big remote CSVs - stream them, never hold 1M rows in RAM."""
        p = grab(url, dest)
        rows, hdr = [], None
        with open(p, "rb") as f:                 # one open, binary, decode per line
            for raw in f:
                line = raw.decode("utf-8", "replace").rstrip("\r\n")
                if not line:
                    continue
                if hdr is None:
                    hdr = next(csv.reader([line]))
                    continue
                rec = dict(zip(hdr, next(csv.reader([line]))))
                a, b = (rec.get(i_c) or "").strip(), (rec.get(o_c) or "").strip()
                if a and b:
                    rows.append((a, b))
                    if len(rows) >= cap:
                        break
        return rows

    CODE_FENCE = re.compile(r"```")
    # Translated/curated data contains the ORIGINAL model refusing. Training on those
    # teaches the uncensored model to refuse - the opposite of the goal.
    REFUSAL = re.compile(
        r"\b(i'?m sorry|i am sorry|i apolog|sorry, but|"
        r"i (?:can'?t|cannot|can not|won'?t|will not)\b|"
        r"i'?m (?:not able|unable|afraid)\b|i am (?:not able|unable)\b|"
        r"as an ai\b|as a language model\b|i must (?:decline|refuse)\b)\b", re.I)
    MARATHI = re.compile(r"(?<![\u0900-\u097F])(आहे|आणि|मला|तुम्ही|काय|मध्ये|साठी|"
                         r"करण्यासाठी|छे|पण|नाही)(?![\u0900-\u097F])")
    FOREIGN = {k: re.compile(v) for k, v in {
        "Bengali": r"[\u0980-\u09FF]", "Gujarati": r"[\u0A00-\u0A7F]",
        "Tamil": r"[\u0A80-\u0AFF]",   "Telugu": r"[\u0B00-\u0B7F]",
        "Kannada": r"[\u0B80-\u0BFF]", "Malayalam": r"[\u0D00-\u0D7F]",
        "Arabic": r"[\u0600-\u06FF]"}.items()}
    DEV = re.compile(r"[\u0900-\u097F]")
    EN_ONLY = re.compile(r"\b(the|and|is|are|of|to|with|for|that|this|you|your|"
                         r"from|have|will|can|not)\b", re.I)
    # 620 chars * 1.45 tok/char + ~15 template ~= 915 tokens, inside MAX_SEQ_LENGTH=1024
    MAX_CHARS = 620
    # ONE floor for every source, no per-source overrides. This used to be 12, with
    # arena allowed down to 6 and casual down to 3 so the row counts would come out
    # right. That measured backwards: casual is 1500 rows of which 0 survive a
    # 20-word floor, and 92% of its replies are under 10 words - the fragment data
    # that produces the 4-word collapse. arena keeps 30% at 20 words. The teacher's
    # own output is 37-49 words, so 20 puts the training data at the teacher instead
    # of two distributions apart from it. Fewer real rows beat padded bad ones.
    MIN_WORDS = 20

    # allow_dev: True = require Devanagari in both, False = require none anywhere,
    # None = require BOTH fields agree, "any" = no script constraint at all.
    def clean(u, a, allow_dev, min_words=MIN_WORDS):
        if not u or not a or CODE_FENCE.search(u) or CODE_FENCE.search(a):
            return None
        if REFUSAL.search(u) or REFUSAL.search(a) or MARATHI.search(u) or MARATHI.search(a):
            return None
        for rx in FOREIGN.values():
            if rx.search(u) or rx.search(a): return None
        if allow_dev is None and bool(DEV.search(u)) != bool(DEV.search(a)):
            return None                                   # one side Devanagari, other not
        if allow_dev == "any":
            pass                                         # mixed is the POINT here
        if allow_dev not in (None, "any") and bool(DEV.search(u) or DEV.search(a)) != allow_dev:
            return None
        if len(u) + len(a) > MAX_CHARS: return None
        if len(a.split()) < min_words: return None        # terse/fragmentary answer
        if len(u.split()) < 3: return None
        return u, a

    random.seed(3407)
    out = {"hinglish": [], "hindi": [], "english": []}
    prov = {}

    # ---- s1: Sujalvc, AAG_anchored only --------------------------------------
    # The 379 GeminiFlash rows are skipped: their replies leak English
    # ("SSY is highly targeted debt scheme for girl child education").
    try:
        raw = grab("https://huggingface.co/datasets/Sujalvc/hinglish-instruct-dataset/"
                   "resolve/main/hinglish_chat_chunk.jsonl",
                   "/kaggle/working/data_cache/sujalvc.jsonl")
        n = 0
        for line in open(raw, encoding="utf-8"):
            try: r = json.loads(line)
            except Exception: continue
            if r.get("source") != "AAG_anchored": continue
            c = clean(r.get("instruction", ""), r.get("output", ""), allow_dev=False)
            if c:
                out["hinglish"].append({"instruction": c[0], "response": c[1],
                                        "src": "sujalvc", "category": "AAG"})
                n += 1
        prov["sujalvc"] = n
    except Exception as e:
        print(f"  sujalvc failed: {type(e).__name__}: {e}")

    # ---- s1: real human Hinglish prompts from chatbot arena -------------------
    try:
        rows = stream_csv("https://huggingface.co/datasets/one-thing/"
                          "chatbot_arena_conversations_hinglish/resolve/main/data.csv",
                          "/kaggle/working/data_cache/arena.csv",
                          "user_hinglish", "assistant_hinglish", 4000)
        n = 0
        for u, a in rows:
            c = clean(u, a, allow_dev=False)
            if c: out["hinglish"].append({"instruction": c[0], "response": c[1],
                                          "src": "arena", "category": "arena"}); n += 1
        prov["arena"] = n
    except Exception as e:
        print(f"  arena failed: {type(e).__name__}: {e}")

    # ---- s1: natural everyday register ---------------------------------------
    try:
        rows = stream_csv("https://huggingface.co/datasets/Abhishekcr448/"
                          "Hinglish-Everyday-Conversations-1M/resolve/main/"
                          "hinglish_conversations.csv",
                          "/kaggle/working/data_cache/casual.csv", "input", "output", 3000)
        n = 0
        for u, a in rows:
            c = clean(u, a, allow_dev=False)
            if c: out["hinglish"].append({"instruction": c[0], "response": c[1],
                                          "src": "casual", "category": "casual"}); n += 1
        prov["casual"] = n
    except Exception as e:
        print(f"  casual failed: {type(e).__name__}: {e}")

    # ---- s2: smangrul Devanagari + zicsx filler -------------------------------
    try:
        raw = grab("https://huggingface.co/datasets/smangrul/hindi_instruct_v1/"
                   "resolve/main/data/train-00000-of-00001.parquet",
                   "/kaggle/working/data_cache/smangrul.parquet")
        import pandas as pd
        df = pd.read_parquet(raw)
        n = 0
        for msgs, cat in zip(df["messages"], df["category"]):
            try:
                u = next(m["content"] for m in msgs if m["role"] == "user")
                a = next(m["content"] for m in msgs if m["role"] == "assistant")
            except (StopIteration, TypeError): continue
            if not (DEV.search(u) and DEV.search(a)): continue      # Devanagari only
            c = clean(u, a, allow_dev=True)
            if c: out["hindi"].append({"instruction": c[0], "response": c[1],
                                       "src": "smangrul", "category": str(cat)}); n += 1
        prov["smangrul_deva"] = n
    except Exception as e:
        print(f"  smangrul failed: {type(e).__name__}: {e}")

    try:   # Devanagari filler - 512k pool, clean, 0.00% Marathi
        raw = grab("https://huggingface.co/datasets/zicsx/indic-align-hindi/resolve/main/"
                   "data/train-00000-of-00026-a3979045df3b161a.parquet",
                   "/kaggle/working/data_cache/zicsx0.parquet")
        df = pd.read_parquet(raw)
        n = 0
        for arr in df["interactions"]:
            try:
                t = list(arr)[0]; u, a = str(t[0]), str(t[1])
            except Exception: continue
            c = clean(u, a, allow_dev=True)
            if c: out["hindi"].append({"instruction": c[0], "response": c[1],
                                       "src": "zicsx", "category": "zicsx"}); n += 1
            if len(out["hindi"]) >= HINDI_TARGET * 2: break
        prov["zicsx"] = n
    except Exception as e:
        print(f"  zicsx failed: {type(e).__name__}: {e}")


    # ---- COMI-LINGUA: 3-expert-annotated, CC-BY-4.0, EMNLP-2025 ---------------
    # Each MT row is triple-parallel: Sentences (code-mixed source),
    # Annotator_1_RH_translation (romanized Hindi), Annotator_1_DH_translation
    # (Devanagari Hindi). This is the ONLY human-expert-annotated Hindi source in
    # the whole search - everything else is machine-translated or LLM-synthesised.
    # Register is news/journalism at ~20 words a sentence, so it supplements the
    # conversational sources rather than replacing them.
    try:
        import csv as _csv
        p = grab("https://huggingface.co/datasets/LingoIITGN/COMI-LINGUA/resolve/main/MT_train.csv",
                 "/kaggle/working/data_cache/comi_MT.csv")
        nr = nd = 0
        with open(p, encoding="utf-8", errors="replace") as f:
            for r in _csv.DictReader(f):
                src_s = (r.get("Sentences") or "").strip()
                rh = (r.get("Annotator_1_RH_translation") or "").strip()
                dh = (r.get("Annotator_1_DH_translation") or "").strip()
                # s1: code-mixed Devanagari prompt -> romanized Hindi reply
                # source is Devanagari, reply is romanized -> allow_dev=None
                # (mixed) then require the REPLY itself to carry no Devanagari.
                # Passing allow_dev=False asked for no Devanagari in EITHER field
                # and rejected all 24,692 rows, which is why comi_rh came out 0.
                if src_s and rh and nr < HINGLISH_TARGET and not DEV.search(rh):
                    q = clean(src_s, rh, allow_dev="any")
                    if q:
                        out["hinglish"].append({"instruction": q[0], "response": q[1],
                                                "src": "comi_rh", "category": "expert"}); nr += 1
                # s2: same source -> Devanagari Hindi reply
                if src_s and dh and nd < HINDI_TARGET:
                    q = clean(src_s, dh, allow_dev=True)
                    if q:
                        out["hindi"].append({"instruction": q[0], "response": q[1],
                                              "src": "comi_dh", "category": "expert"}); nd += 1
        prov["comi_rh"] = nr
        prov["comi_dh"] = nd
    except Exception as e:
        print(f"  comi-lingua failed: {type(e).__name__}: {e}")

    # ---- s3: English anchor ---------------------------------------------------
    # NOT smangrul. Its romanized rows are Hinglish with English loanwords, so
    # filtering them for "English" harvested 511 rows of more Hinglish - an anchor
    # made of the very thing it is supposed to balance. dolly-15k is genuinely
    # human-written English instruction data.
    try:
        p = grab("https://huggingface.co/datasets/databricks/databricks-dolly-15k/"
                 "resolve/main/databricks-dolly-15k.jsonl",
                 "/kaggle/working/data_cache/dolly.jsonl")
        n = 0
        for line in open(p, encoding="utf-8"):
            try: r = json.loads(line)
            except Exception: continue
            q = clean((r.get("instruction") or "").strip(),
                      (r.get("response") or "").strip(),
                      allow_dev=False)
            if q:
                out["english"].append({"instruction": q[0], "response": q[1],
                                       "src": "dolly", "category": "english"}); n += 1
        prov["dolly"] = n
    except Exception as e:
        print(f"  dolly failed: {type(e).__name__}: {e}")

    # ---- mix, write, and REPORT THE THING THAT MATTERS -----------------------
    def dump(name, data, n_target, quotas=None):
        """Exact per-source quotas, NOT shuffle-then-cap.

        Shuffling the pool and taking the first N silently diluted the scarce good
        sources: the hindi pool was 18,000, we took a random 6,000, and smangrul
        (conversational) fell from 2,185 to 751 while 6,569 zicsx rows were thrown
        away. Shortage of a good source is a fact to surface, not to average away.
        """
        if quotas:
            by = {}
            for r in data:
                by.setdefault(r["src"], []).append(r)
            sel, short = [], []
            for s_name, want in quotas.items():
                have = by.get(s_name, [])
                random.shuffle(have)
                sel.extend(have[:want])
                if len(have) < want:
                    short.append(f"{s_name} {len(have)}/{want}")
            if short:
                print(f"  !! {name} short of: {', '.join(short)}")
        else:
            random.shuffle(data)
            sel = data[:n_target]
        random.shuffle(sel)
        p = f"/kaggle/working/{name}"
        with open(p, "w", encoding="utf-8") as f:
            for r in sel:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        return sel

    print("\n  source            rows kept")
    for k, v in prov.items():
        print(f"  {k:17s} {v:>8,}")

    targets = {"s1_hinglish": ("hinglish", HINGLISH_TARGET),
               "s2_hindi": ("hindi", HINDI_TARGET),
               "s3_english": ("english", ENGLISH_TARGET)}
    print("\n  session        rows   median-reply-words  sources")
    quotas = {
        "s1_hinglish": {"sujalvc": 4500, "arena": 1500, "casual": 1000, "comi_rh": 1500},
        "s2_hindi":    {"smangrul": 6000, "comi_dh": 2000, "zicsx": 3000},
        "s3_english":  {"dolly": ENGLISH_TARGET},
    }
    for sess, (b, t) in targets.items():
        sel = dump(f"train_{sess}.jsonl", out[b], t, quotas.get(sess))
        mw = sorted(len(r["response"].split()) for r in sel)
        med = mw[len(mw)//2] if mw else 0
        srcs = {}
        for r in sel: srcs[r["src"]] = srcs.get(r["src"], 0) + 1
        print(f"  {sess:13s} {len(sel):>6,}   {med:>10}          {srcs}")
        if med < 20:
            print(f"  !! {sess} median reply is {med} words - that is the terse-fragment "
                  f"failure mode, not chat. Check the sources above.")

    # script + contamination audit
    print("\n--- audit ---")
    for b, data in out.items():
        if not data: continue
        dev = sum(1 for r in data if DEV.search(r["instruction"] + r["response"]))
        mar = sum(1 for r in data if MARATHI.search(r["instruction"] + r["response"]))
        eng = sum(1 for r in data if len(EN_ONLY.findall(r["response"])) >= 6)
        print(f"  {b:9s} n={len(data):>6,}  devanagari={dev:>6,}  marathi={mar}  english_reply={eng}")

In [ ]:
import os, glob, json, re
DATA_REPO = "kumarsujitroy/lfm25-teacher-data"
CANDIDATES = ([f"/kaggle/working/{DATA_FILE}"]
              + [f"/kaggle/input/hinglish-sft/{DATA_FILE}"]
              + glob.glob(f"/kaggle/input/*/{DATA_FILE}"))
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)

if DATA_PATH is None:
    import urllib.request
    tok = os.environ.get("HF_TOKEN", "").strip()
    urls = ([f"https://huggingface.co/datasets/{DATA_REPO}/resolve/main/{DATA_FILE}"]
            + [f"https://raw.githubusercontent.com/SujitRoy/hindi-finetune/main/{DATA_FILE}"])
    for url in urls:
        try:
            h = {"Authorization": f"Bearer {tok}"} if (tok and "huggingface.co" in url) else {}
            req = urllib.request.Request(url, headers=h)
            with urllib.request.urlopen(req, timeout=600) as r:
                payload = r.read()
            DATA_PATH = f"/kaggle/working/{DATA_FILE}"
            os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
            with open(DATA_PATH, "wb") as f:
                f.write(payload)
            print(f"fetched {len(payload):,} bytes from {url.split('/')[2]}")
            break
        except Exception as e:
            print(f"  {url.split('/')[2]}: {type(e).__name__} {getattr(e, 'code', '')}")
    else:
        raise RuntimeError(f"could not fetch {DATA_FILE} from anywhere")

rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8")]
srcs = {s: sum(1 for r in rows if r["src"] == s) for s in {r["src"] for r in rows}}
lang = {}
for r in rows:
    lang[r.get("lang", "?")] = lang.get(r.get("lang", "?"), 0) + 1
print(len(rows), "rows |", srcs)
print("languages:", lang)
dev = sum(1 for r in rows if any("\u0900" <= c <= "\u097F"
                                for c in r["instruction"] + r["response"]))
print(f"{dev} rows contain Devanagari")

# ==========================================================================
# LANGUAGE ASSERTION. Marathi is written in Devanagari, so every existing
# script check passes a poisoned corpus: "hindi rows are >=55% Devanagari" is
# true of Marathi, and 4.8% of v6's Hindi rows were exactly that.
#
# The FIX is a source blacklist and it runs LOCALLY in teach/build_release.py,
# where the prompt pools live (1,245/1,245 Marathi rows = 100% attributable to
# the `adaption` machine-translated pool). What this cell does is prove the
# blacklist worked before spending 6 GPU-hours. No pool files, no network.
#
# The marker list is calibrated, not guessed. Measured against
# ds_hindi_devanagari.jsonl (6,000 known-Hindi texts): करते and सकते hit 479
# and 660 of them because they are ordinary Hindi inflections (करते हैं, बता
# सकते हैं); Marathi uses करतो / शकतो. Unanchored forms match inside Hindi
# words (मला in दिल्ली, काय in कार्य, छे in अच्छे), so both sides are anchored.
# ==========================================================================
_MAR = re.compile(r"(?<![\u0900-\u097f])(आहे|आणि|तुम्ही|तुम्हाला|पण|मला|माझे|माझा|माझी|"
                  r"झाले|झाला|काय|कोणता|कोणती|म्हणजे|मुळे|याच|दिला|दिली|सकतो|असतो|आहोत)"
                  r"(?![\u0900-\u097f])")
_HIN = re.compile(r"(?<![\u0900-\u097f])(है|हैं|हूँ|और|क्या|में|से|के|लिए|नहीं|हुआ|होता|"
                  r"करना|किया|गया|मिला|दिया|सकता|सकती|रहा|रही)(?![\u0900-\u097f])")
_FOREIGN = re.compile("[\u0980-\u09ff\u0a80-\u0aff\u0b80-\u0dff\u3040-\u30ff"
                      "\u4e00-\u9fff\uac00-\ud7af\u0600-\u06ff]")

_mar = [r for r in rows
        if len(set(_MAR.findall(r["response"]))) > len(set(_HIN.findall(r["response"])))]
_foreign = [r for r in rows
            if _FOREIGN.search(r["instruction"]) or _FOREIGN.search(r["response"])]
_nonhh = [r for r in rows if r.get("lang") not in ("hindi", "hinglish")]

print(f"language assertion on {len(rows):,} rows:")
print(f"  marathi-dominated : {len(_mar):,}")
print(f"  foreign script    : {len(_foreign):,}")
print(f"  not hindi/hinglish: {len(_nonhh):,}")
for r in (_mar + _foreign + _nonhh)[:3]:
    print("    e.g.", r.get("lang"), "|", r["instruction"][:46], "->", r["response"][:46])

# Hard stops. Each one is a 6-hour run wasted and a model that still sounds wrong.
assert not _mar, (
    f"{len(_mar)} Marathi-dominated rows in {DATA_FILE}. This file is supposed to be"
    " the OUTPUT of teach/build_release.py, which blacklists the source. Rebuild the"
    " release locally; do NOT widen this regex to make the run go through.")
assert not _nonhh, (
    f"{len(_nonhh)} rows are not hindi/hinglish (scope is those two only)."
    " Rebuild with teach/build_release.py.")
# foreign script is scraped garbage in small volume, filtered here rather than
# failed on: 97 rows measured, and it cannot be fixed by re-pulling a source.
if _foreign:
    rows = [r for r in rows if r not in _foreign]
    print(f"  dropped {len(_foreign)} foreign-script rows; continuing")
assert len(rows) > 40_000, f"only {len(rows)} rows survive; expected ~65,721 for v8 - a fetch truncation"
_hi = sum(1 for r in rows if r["lang"] == "hindi")
_forms = len({w for r in rows for w in re.findall("[\\u0900-\\u097f]+", r["response"])})
print(f"FINAL: {len(rows):,} rows | hindi {_hi:,} hinglish {len(rows)-_hi:,} | "
      f"distinct Devanagari forms {_forms:,}")

## 1c. Teacher generation (optional, hours)

Every Hindi instruction corpus on the Hub is machine-translated or LLM-synthesised, so
the student learns the register of those translations rather than the language. The
teacher writes the surface form correctly - measured, not assumed: asked to translate
well-formed Hindi into romanized Hinglish it produced `Agar vah kalaakaar hai, to ek
chhotaa chit ... diya ja sakta hai jo uski ruchi ko dikhaye` with correct verb
agreement. Asked to *generate* Hindi freely, the same model does worse. So we let it
generate, and filter hard.

**The filter is the deliverable, not the prompt.** The teacher's 7/7 probe answers were
already clean; the risk is a bad row slipping into the mix. Four rejects, and the
length floor is the important one: the student collapses below 30 words, the teacher
writes 37-49, so a 30-word floor passes real rows and kills the collapse at the source.

**Where this runs.** chdir to /tmp before the teacher is fetched. Last time the 16.4 GB
of Qwen weights landed in /kaggle/working and left 0.9 GB, which killed the GGUF export
at the end of training. /tmp has over a terabyte and the weights are disposable.

In [ ]:
# Optional. Set RUN_GEN = False to skip.
# Generation now runs on this box against the API: 4,000 rows/h, no GPU,
# no 12 h session wall. See teach/README.md. Do not re-run it on Kaggle.
RUN_GEN   = False
TEACHER   = "Qwen/Qwen3-8B"      # 16.4 GB. "unsloth/Llama-3.2-3B-Instruct" is 6.4 GB and faster.
N_PER_PASS = 800                 # per language
# The teacher writes 47 words median. Hindi is 1.31 tok/char, so that needs ~310
# tokens. MAX_NEW was 220 and cut EVERY row mid-word; 96% then failed to end on
# terminal punctuation. 380 is headroom over a 30-50 word answer.
MAX_NEW   = 380
BATCH     = 8
PUSH_EVERY_S = 300               # Kaggle local storage is not durable
TIME_BUDGET_H = 9.0              # exit and push cleanly instead of dying mid-batch
# v2, not the v1 path: 96 truncated rows from the earlier run are still published at
# teacher_gen.jsonl and must never reach a training cell.
OUT_JSONL = "/tmp/teacher_gen_v2.jsonl"

if not RUN_GEN:
    print("RUN_GEN=False - skipping")
else:
    import os, re, json, time, hashlib, shutil, urllib.request, urllib.error, torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from huggingface_hub import HfApi

    os.chdir("/tmp")
    print(f"cwd {os.getcwd()}  {shutil.disk_usage('/tmp').free/2**30:.0f} GB free")

    SMALL_FILES = ["config.json", "generation_config.json", "tokenizer.json",
                   "tokenizer_config.json", "vocab.json", "merges.txt",
                   "added_tokens.json", "special_tokens_map.json"]

    def fetch_repo(repo, dest):
        def get(fn):
            # A token is harmless on a public repo and required on a private one. The
            # s1/s2 repos were made private, so any cell that READS a base model back
            # through this function must send it.
            tok = os.environ.get("HF_TOKEN", "").strip()
            hdr = {"Authorization": f"Bearer {tok}"} if tok else {}
            with urllib.request.urlopen(
                    urllib.request.Request(
                        f"https://huggingface.co/{repo}/resolve/main/{fn}",
                        headers=hdr), timeout=600) as r:
                return r.read()
        os.makedirs(dest, exist_ok=True)
        have = set(os.listdir(dest))
        idx = "model.safetensors.index.json"
        if idx not in have:
            open(os.path.join(dest, idx), "wb").write(get(idx))
            have.add(idx)
        need = list(SMALL_FILES)
        try:
            need += sorted(set(json.load(open(os.path.join(dest, idx)))["weight_map"].values()))
        except Exception:
            need.append("model.safetensors")
        total = 0
        for fn in need:
            dst = os.path.join(dest, fn)
            if fn in have and os.path.getsize(dst) > 0:
                continue
            try:
                body = get(fn)
            except urllib.error.HTTPError as e:
                print(f"  skip {fn} ({e.code})")
                continue
            with open(dst, "wb") as f:
                f.write(body)
            total += len(body)
            print(f"  {fn:34s} {len(body):>12,}", flush=True)
        print(f"  fetched {total/1e9:.1f} GB -> {dest}")
        return dest

    DEV_SYS = ("Aap ek helpful assistant ho. User ke sawaal ka jawab POORI tarah se, saaf "
               "aur natural Devanagari Hindi mein likho. English se seedha translation mat "
               "karo - waise likho jaise ek native Hindi speaker likhta hai. Markdown ya "
               "bullet points use mat karo, plain paragraph likho. Jawab 30 se 50 shabd ka "
               "likho. Isse zyada lamba mat likho.")
    HING_SYS = ("Aap ek helpful assistant ho. User Hinglish mein likhta hai (romanized Hindi "
                "+ English mix). Aap bhi isi tarah romanized Hinglish mein jawab do. Poora aur "
                "sahi jawab do. Varanak ke liye 'hai' aur 'hain' ka sahi prayog karo, 'h' ya "
                "'kr' jaise shortcut mat likho. Markdown ya bullets use mat karo. Jawab 30 se "
                "50 shabd ka likho.")

    BAD   = re.compile(r"(ai assistant|ai model|as an ai|मैं एक एआई|\bh\b(?!\w)|\bkr\b|\bkro\b)", re.I)
    REFU  = re.compile(r"(i am sorry|i cannot|i can't|maine pucha|mujhe nahi pata|"
                       r"i'm not able|as an ai)", re.I)
    # A reply that does not land on terminal punctuation was truncated, not finished.
    # Length filters cannot catch this; only this check can.
    # The ASCII period MUST be in this set: Hinglish replies end in "." and an earlier
    # version omitted it, which would have rejected 100% of the Hinglish half as
    # "truncated". Caught by running accept() against known-good samples.
    ENDS  = re.compile(r"[।.?!\"'”’)\]]\s*$")

    def accept(resp, want_dev):
        w = len(resp.split())
        if w < 30:                          return False, "short"
        if w > 160:                         return False, "ramble"
        if not ENDS.search(resp):           return False, "truncated"
        if BAD.search(resp):                return False, "shorthand"
        if REFU.search(resp):               return False, "refusal"
        toks = resp.split()
        tri = [tuple(toks[j:j + 3]) for j in range(len(toks) - 2)]
        if tri and len(set(tri)) / len(tri) < 0.92:
            return False, "repetitive"
        dev = sum("ऀ" <= c <= "ॿ" for c in resp) / max(1, len(resp))
        if want_dev and dev < 0.55:         return False, "not_devanagari"
        if not want_dev and dev > 0.35:     return False, "went_devanagari"
        return True, ""

    CACHED = "/kaggle/working/data_cache/dolly.jsonl"
    DOLLY_URL = ("https://huggingface.co/datasets/databricks/databricks-dolly-15k/"
                 "resolve/main/databricks-dolly-15k.jsonl")
    dl = CACHED if os.path.exists(CACHED) else "/tmp/dolly.jsonl"
    if not os.path.exists(dl):
        with urllib.request.urlopen(DOLLY_URL, timeout=900) as r, open(dl, "wb") as f:
            f.write(r.read())
    print(f"dolly: {dl}")

    instrs = []
    for line in open(dl, encoding="utf-8"):
        try: q = json.loads(line)["instruction"].strip()
        except Exception: continue
        if not q: continue
        w = len(q.split())
        if not (3 <= w <= 40): continue
        if q.lower().rstrip(" ?.").split()[-1] in ("or","and","is","are","the","a","an"): continue
        if re.match(r"^(is|are|was|were|do|does|did|can|could|will|would|should|has|have|had)\b",
                    q, re.I) and w <= 6: continue
        instrs.append(q)
    print(f"dolly: {len(instrs)} usable instructions")

    done = set()
    if os.path.exists(OUT_JSONL):
        for line in open(OUT_JSONL, encoding="utf-8"):
            try: done.add(json.loads(line)["key"])
            except Exception: pass
        print(f"resuming: {len(done)} rows already generated")

    DEST = "/tmp/teacher"
    fetch_repo(TEACHER, DEST)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16)
    tk = AutoTokenizer.from_pretrained(DEST)
    if tk.pad_token_id is None:
        tk.pad_token = tk.eos_token
    md = AutoModelForCausalLM.from_pretrained(DEST, quantization_config=bnb,
                                             device_map="auto").eval()

    @torch.no_grad()
    def gen_batch(qs, sys_p):
        """Decode a whole batch at once. Left padding is required or generation
        continues from pad tokens. This is the teacher's tokenizer, not the
        student's, so padding_side here does not touch the student path."""
        tk.padding_side = "left"
        texts = [tk.apply_chat_template(
                     [{"role": "system", "content": sys_p},
                      {"role": "user", "content": q}],
                     add_generation_prompt=True, enable_thinking=False, tokenize=False)
                 for q in qs]
        enc = tk(texts, return_tensors="pt", padding=True).to(md.device)
        o = md.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                        pad_token_id=tk.pad_token_id)
        out = o[:, enc["input_ids"].shape[1]:]
        return [x.strip() for x in tk.batch_decode(out, skip_special_tokens=True)]

    def push():
        HfApi().upload_file(path_or_fileobj=OUT_JSONL, path_in_repo="teacher_gen_v2.jsonl",
                            repo_id=REPO, revision=BRANCH, commit_message=f"teacher gen {int(time.time())}")

    kept = {"hindi": 0, "hinglish": 0}
    dropped = {}
    last_push, t_start = time.time(), time.time()
    fh = open(OUT_JSONL, "a", encoding="utf-8")
    # Interleave the two languages. Running all Hindi then all Hinglish starves
    # Hinglish completely if the session wall arrives during the Hindi half.
    SYS = {"hindi": (True, DEV_SYS), "hinglish": (False, HING_SYS)}
    pending = {t: [q for q in instrs
                   if hashlib.sha1(f"{t}|{q}".encode()).hexdigest()[:16] not in done]
               for t in ("hindi", "hinglish")}
    cursor = {t: 0 for t in pending}
    over = False

    while not over:
        if all(kept[t] >= N_PER_PASS or cursor[t] >= len(pending[t])
               for t in ("hindi", "hinglish")):
            break                      # both languages exhausted; do not spin
        for tag in ("hindi", "hinglish"):
            want_dev, sys_p = SYS[tag]
            if kept[tag] >= N_PER_PASS or cursor[tag] >= len(pending[tag]):
                continue
            b = cursor[tag]
            cursor[tag] += BATCH
            if time.time() - t_start > TIME_BUDGET_H * 3600:
                print(f"  time budget reached at {kept}", flush=True)
                over = True
                break
            chunk = pending[tag][b:b + BATCH]
            if not chunk:
                continue
            try:
                answers = gen_batch(chunk, sys_p)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                answers = []
                for q in chunk:              # fall back to one at a time
                    try: answers += gen_batch([q], sys_p)
                    except Exception: pass
            except Exception as e:
                print(f"  gen error {type(e).__name__}: {e}", flush=True)
                answers = []
            for q, a in zip(chunk, answers):
                if kept[tag] >= N_PER_PASS:
                    break
                key = hashlib.sha1(f"{tag}|{q}".encode()).hexdigest()[:16]
                ok, why = accept(a, want_dev)
                if ok:
                    fh.write(json.dumps({"instruction": q, "response": a, "src": "teacher",
                                         "lang": tag, "key": key}, ensure_ascii=False) + "\n")
                    fh.flush()
                    kept[tag] += 1
                    if kept[tag] % 10 == 0:
                        el = time.time() - t_start
                        print(f"[{time.strftime('%H:%M:%S')}] {tag} {kept[tag]}/{N_PER_PASS}  "
                              f"{el/60:.0f}m  {el/max(1,sum(kept.values())):.1f}s/row  "
                              f"drop={dropped}", flush=True)
                else:
                    dropped[why] = dropped.get(why, 0) + 1
            if time.time() - last_push >= PUSH_EVERY_S:
                push(); last_push = time.time()
                print(f"  pushed {kept}", flush=True)

    fh.close()
    push()
    print(f"\nkept    : {kept}")
    print(f"dropped : {dropped}")
    print(f"file    : {OUT_JSONL}")

In [ ]:
import torch

MAX_SEQ_LENGTH = 4096      # vendor recipe. At 1024 a long Hindi answer is silently
DTYPE = torch.float16      # T4 is sm_75, no bf16
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DTYPE)
# truncated; the 620-char data cap made that invisible. 4096 is what Liquid ships.

## 2. Local tokenizer

Two upstream problems, fixed here rather than worked around:

1. The abliterated base repo declares `tokenizer_class: TokenizersBackend`, a
   **transformers 5.x** class. This notebook runs 4.57, so it cannot be resolved.
2. **Neither** LFM repo embeds the chat template in `tokenizer_config.json`, so
   transformers fetches it from the Hub - and that lookup 404s.

So we build a local, self-contained tokenizer dir: 4.x-compatible class, template
embedded inline, and a real `model_max_length` (upstream ships `float('inf')` as an
integer). After this cell nothing touches the Hub for the tokenizer.

Verified: vocab (64,400) and merges (63,683) are byte-identical to the abliterated
repo, and this renders the same 29-token 2-turn string the data was built against.

In [ ]:
import os
import json

TOKENIZER_REPO = "LiquidAI/LFM2.5-1.2B-Instruct"
TOK_DIR = "/kaggle/working/tok"
os.makedirs(TOK_DIR, exist_ok=True)

for fn in ["tokenizer.json", "tokenizer_config.json",
           "chat_template.jinja", "special_tokens_map.json"]:
    dst = os.path.join(TOK_DIR, fn)
    if not os.path.exists(dst):
        url = f"https://huggingface.co/{TOKENIZER_REPO}/resolve/main/{fn}"
        import urllib.request
        with urllib.request.urlopen(url, timeout=180) as r, open(dst, "wb") as f:
            f.write(r.read())

cfg_path = os.path.join(TOK_DIR, "tokenizer_config.json")
cfg = json.load(open(cfg_path))
cfg["chat_template"] = open(os.path.join(TOK_DIR, "chat_template.jinja")).read()
cfg["model_max_length"] = MAX_SEQ_LENGTH
cfg["tokenizer_class"] = "PreTrainedTokenizerFast"
json.dump(cfg, open(cfg_path, "w"), ensure_ascii=False)

from transformers import AutoTokenizer
_chk = AutoTokenizer.from_pretrained(TOK_DIR)
print("tokenizer ready |", len(_chk.get_vocab()), "tokens | max_len", _chk.model_max_length)
print("probe:", repr(_chk.apply_chat_template(
    [{"role": "user", "content": "kya haal hai?"},
     {"role": "assistant", "content": " sab badhiya"}],
    tokenize=False, add_generation_prompt=False)))

## 1d. Continued pre-training on raw Hindi (CPT)

Liquid's own language-adaptation recipe is two stages, and we were only running the
second one. From `cpt_translation_with_unsloth.ipynb`:

> "perform **continued pre-training** on Korean Wikipedia data, **followed by**
> instruction fine-tuning on Korean translation examples. **This approach is ideal
> for adapting models to specific languages**"

Their mechanism for the new script is in the LoRA call:

> "We now add `embed_tokens` and `lm_head` to allow the model to learn **out of
> distribution data**"

with `embedding_learning_rate` set 2-10x below `learning_rate`, and
`use_rslora=True` (True for CPT, False for SFT - the two notebooks differ on this
deliberately).

Session 2 previously went straight to SFT on 7,185 instruction pairs, so the model
had never seen a single raw-text pass over Hindi. This stage supplies it, from
Hindi Wikipedia, in exactly their format.

**Why not `LiquidAI/LFM2.5-1.2B-Base`** as the CPT starting point, as the notebook
uses: that model is not abliterated, and the abliteration is the reason for this
whole exercise. Chaining CPT from the abliterated repo keeps it. Everything else in
this cell follows the vendor cell for cell.

In [ ]:
# --- Weights & Biases. A text log of 66 loss points is not a graph, and the one
# thing this project kept needing was to SEE the eval-loss minimum rather than
# infer it from a table. Optional: with no secret the run continues and the curve
# prints to the log, so it never blocks a session.
import os

WANDB_ON = False
try:
    import wandb
    _k = ""
    try:
        from kaggle_secrets import UserSecretsClient
        _k = UserSecretsClient().get_secret("WANDB_API_KEY") or ""
    except Exception:
        pass
    _k = (_k or os.environ.get("WANDB_API_KEY", "")).strip()
    if _k:
        wandb.login(key=_k)
        WANDB_ON = True
    else:
        print("no WANDB_API_KEY secret - training anyway, curves print to the log")
except Exception as e:
    print(f"wandb unavailable ({type(e).__name__}) - curves print to the log")

WANDB_PROJECT = "lfm25-hindi"

## 3. Train on BOTH T4s - DDP via `torchrun`

The notebook used to run one process on one card, and the second 15 GB T4 sat idle.
That was blamed on a hard block in Unsloth, but `patch_sft_trainer_tokenizer()` - the
only place the `does not support multi GPU` raise lives - has **no call site** in the
installed `unsloth-2026.9.14` wheel, so it is never installed on `SFTTrainer.train`.
The documented path,
<https://unsloth.ai/docs/basics/multi-gpu-training-with-unsloth>, is Distributed Data
Parallel launched with `torchrun`, and that is what the next three cells do:

1. **the config cell** renders every conversation, length-groups the batches, saves the
   split and `ddp_config.json`, and exposes `RUN_CPT` / batch sizes.
2. **the `%%writefile` cell** writes `train_ddp.py` - the whole training program - to
   `/kaggle/working`.
3. **the launch cell** runs it under
   `python -m torch.distributed.run --standalone --nproc_per_node=2`: one process per T4.

Each rank loads the model on its own card (Unsloth reads `LOCAL_RANK`/`WORLD_SIZE` and
pins the device), trains on its shard of the data, and DDP all-reduces the LoRA
gradients once per optimizer step. Two T4s therefore double rows/step at the **same
effective batch of 16** - no hyper-parameter changes. Rank 0 alone owns the Hub: the
wall-clock checkpoints, the merged `cpt-hindi` push, and the final merge -> GGUF.

Model targets, `use_rslora` and the response masking are all inside `train_ddp.py`,
so there is one copy of the training logic rather than a notebook cell and a script
that can drift apart.

In [ ]:
import json, random
from datasets import Dataset

# ---- run config for train_ddp.py. This cell is the one place a run is set.
RUN_CPT    = True                 # Liquid stage 1: install Hindi before SFT
CPT_STEPS  = 1500                 # 1500 steps x 16 rows = 24,000 docs, ~2.5 h on 2 cards
CPT_SEQ    = 1024                 # measured: median 528, p90 987 tokens
CPT_LR     = 5e-5                 # Liquid cpt_text_completion / cpt_translation
CPT_EMB_LR = 1e-5                 # "2 to 10x smaller than learning_rate"
CPT_R      = 128                  # "Add for continual pretraining" uses 128; SFT uses 16
CPT_ALPHA  = 32
CPT_FILE   = "cpt_sample.jsonl"
CPT_BS, CPT_ACC = 1, 8            # 1 x 8 x 2 GPUs = 16 rows/step
SFT_BS, SFT_ACC = 2, 4            # 2 x 4 x 2 GPUs = 16 rows/step, same budget as v7
SFT_BASE   = "/kaggle/working/cpt-merged" if RUN_CPT else BASE_REPO

# ---- SFT split: render every conversation, then group batches by length. The
# script only loads what this cell saves, so this order is what training sees.
def _pair(r):
    return [{"role": "user", "content": r["instruction"]},
            {"role": "assistant", "content": r["response"]}]

rows = [{**r, "messages": r.get("messages") or _pair(r)} for r in rows]

def to_text(batch):
    convs = batch["messages"]
    return {"text": [tokenizer.apply_chat_template(c, tokenize=False,
                                                   add_generation_prompt=False)
                            .removeprefix(tokenizer.bos_token or "")
                     for c in convs]}

ds = Dataset.from_list(rows).map(to_text, batched=True, remove_columns=list(rows[0].keys()))
split = ds.train_test_split(test_size=0.02, seed=3407)

_lens = [len(t) for t in split["train"]["text"]]
_perm = sorted(range(len(_lens)), key=lambda i: _lens[i])
_STEP = 16                       # per_device 2 x grad_accum 4 x 2 GPUs = one optimizer step
_steps = [_perm[i:i + _STEP] for i in range(0, len(_perm) - len(_perm) % _STEP, _STEP)]
random.Random(3407).shuffle(_steps)
_order = [j for st in _steps for j in st]
if len(_perm) % _STEP:           # keep every row; the ragged tail goes at the end
    _order += _perm[len(_steps) * _STEP:]
assert sorted(_order) == list(range(len(_lens))), "grouping lost or duplicated rows"
split["train"] = split["train"].select(_order)

def _padded(lens, order):
    idx = [order[i] for i in range(len(order) - len(order) % 2)]
    return sum(max(lens[idx[i]], lens[idx[i + 1]]) for i in range(0, len(idx), 2)) * 2

_grouped = _padded(_lens, _order)
_r = random.Random(99); _sh = list(_order); _r.shuffle(_sh)
_shuffled = _padded(_lens, _sh)
print(f"length grouping: {len(_steps):,} steps x {_STEP} rows")
print(f"  padded positions/epoch: grouped {_grouped/1e6:,.1f}M vs shuffled {_shuffled/1e6:,.1f}M "
      f"-> {100*(1-_grouped/_shuffled):.0f}% less padding, same {len(_order):,} rows")
if _grouped >= _shuffled:
    print("  WARNING: grouping did NOT reduce padding - the order is not length-grouped")
print(f"SFT split: train {len(split['train']):,} eval {len(split['test']):,} rows")
assert len(split["train"]) > 40_000, "the split is far smaller than v8 - a fetch truncation"
split.save_to_disk("/kaggle/working/sft_split")

# ---- the script reads this, so a value can never drift between notebook and training
CFG = {
    "REPO": REPO, "BRANCH": BRANCH, "WIP": WIP,
    "BASE_REPO": BASE_REPO, "SFT_BASE": SFT_BASE,
    "DATA_REPO": DATA_REPO, "TOK_DIR": TOK_DIR,
    "MAX_SEQ_LENGTH": MAX_SEQ_LENGTH, "DTYPE": "float16",
    "MAX_STEPS": MAX_STEPS, "LEARNING_RATE": LEARNING_RATE,
    "CHECKPOINT_EVERY_S": CHECKPOINT_EVERY_S,
    "RUN_CPT": RUN_CPT, "CPT_STEPS": CPT_STEPS, "CPT_SEQ": CPT_SEQ,
    "CPT_LR": CPT_LR, "CPT_EMB_LR": CPT_EMB_LR, "CPT_R": CPT_R, "CPT_ALPHA": CPT_ALPHA,
    "CPT_FILE": CPT_FILE, "CPT_BS": CPT_BS, "CPT_ACC": CPT_ACC,
    "SFT_BS": SFT_BS, "SFT_ACC": SFT_ACC, "SFT_SPLIT": "/kaggle/working/sft_split",
    "WANDB_ON": WANDB_ON, "WANDB_PROJECT": WANDB_PROJECT,
}
json.dump(CFG, open("/kaggle/working/ddp_config.json", "w"), indent=1)
print("ddp_config.json written - RUN_CPT =", RUN_CPT)

In [ ]:
%%writefile /kaggle/working/train_ddp.py
#!/usr/bin/env python3
"""Hindi CPT + SFT for LFM2.5-1.2B under Distributed Data Parallel.

Launched by the notebook - one process per GPU - as:

    python -m torch.distributed.run --standalone --nproc_per_node=2 train_ddp.py --stage cpt
    python -m torch.distributed.run --standalone --nproc_per_node=2 train_ddp.py --stage sft

Why DDP and not the old single process. one card trained and the second T4 sat
idle. The old notebook's comment blamed a hard block in Unsloth -
`tokenizer_utils.py:1813` "does not support multi GPU". That block is dead code
in the installed release: `patch_sft_trainer_tokenizer()` is defined but has no
call site in unsloth-2026.9.14, so it is never installed on `SFTTrainer.train`.
The documented path (unsloth.ai/docs/basics/multi-gpu-training-with-unsloth) is
DDP via `torchrun`, and Unsloth's own device planner pins each rank to its card
as soon as `WORLD_SIZE > 1`. Two T4s then train on distinct shards and all-reduce
the LoRA gradients once per optimizer step, doubling rows/step at the same
effective batch.

`device_map` is deliberately NOT passed: unsloth's planner reads LOCAL_RANK and
resolves `{"": "cuda:<rank>"}` per process, which is the one placement that keeps
the two ranks off each other's card. Only rank 0 writes to the Hub - the
wall-clock checkpoint callback, the merged CPT push, the final merge -> GGUF -
so two processes never race on one revision.

Every number comes from /kaggle/working/ddp_config.json, written by the notebook,
so the notebook stays the single place a run is configured.
"""
import argparse
import gc
import json
import os
import time
import urllib.request

CFG_PATH = os.environ.get("DDP_CONFIG", "/kaggle/working/ddp_config.json")
CFG = json.load(open(CFG_PATH, encoding="utf-8"))

LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))
IS_MAIN = LOCAL_RANK == 0

# unsloth first, before any transformers import in this process, or it warns and
# skips its patches. The model is placed by unsloth's distributed planner from
# LOCAL_RANK/WORLD_SIZE - no explicit device_map, which would fight the planner.
import torch  # noqa: E402  (after reading CFG; torch does not touch transformers)

torch.cuda.set_device(LOCAL_RANK)

from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments  # noqa: E402
from unsloth.chat_templates import train_on_responses_only  # noqa: E402
from transformers import AutoTokenizer, TrainerCallback  # noqa: E402
from datasets import Dataset, load_from_disk  # noqa: E402
from peft import PeftModelForCausalLM  # noqa: E402

DTYPE = torch.float16 if CFG["DTYPE"] == "float16" else torch.bfloat16


def log(msg):
    print(f"[rank {LOCAL_RANK}/{WORLD_SIZE}] {msg}", flush=True)


class TimeCheckpoint(TrainerCallback):
    """Push the adapter + tokenizer to the Hub every `every_s` wall-clock seconds.

    Rank 0 only: the callback runs on every process, and two writers to one repo
    revision race. `state.is_world_process_zero` is the DDP-safe form of
    `local_rank == 0`.
    """

    def __init__(self, model, repo, every_s, tokenizer=None, revision=None):
        self.model, self.repo, self.every = model, repo, every_s
        self.rev, self.tok, self.last = revision, tokenizer, time.time()

    def on_step_end(self, args, state, control, **kw):
        if not state.is_world_process_zero:
            return control
        if time.time() - self.last >= self.every:
            tag = f"ckpt-s{state.global_step // 100 * 100:05d}"
            self.model.push_to_hub(self.repo, save_name=tag, revision=self.rev, private=True)
            if self.tok is not None:
                self.tok.push_to_hub(self.repo, save_name=tag, revision=self.rev, private=True)
            log(f"pushed {tag} @ step {state.global_step}")
            self.last = time.time()
        return control


def merge_if_adapter(model):
    """A repo publishing merged weights AND an adapter makes from_pretrained
    attach the adapter; get_peft_model then refuses with "You already added LoRA
    adapters". Merge it in first instead of applying it twice."""
    if isinstance(model, PeftModelForCausalLM):
        log("base carries an adapter - merging it before attaching a new one")
        model = model.merge_and_unload()
    return model


def fetch(url, dest):
    tok = os.environ.get("HF_TOKEN", "").strip()
    hdr = {"Authorization": f"Bearer {tok}"} if tok else {}
    with urllib.request.urlopen(urllib.request.Request(url, headers=hdr), timeout=600) as r:
        body = r.read()
    with open(dest, "wb") as f:
        f.write(body)
    return dest


def push_license(repo, revision):
    """Liquid Open License v1.0 has 16 redistribution clauses; the text travels
    with the weights, so it goes onto the branch that carries them."""
    from huggingface_hub import upload_file
    url = "https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct/resolve/main/LICENSE"
    with urllib.request.urlopen(url, timeout=180) as r:
        lic = r.read()
    with open("/tmp/CPT_LICENSE", "wb") as f:
        f.write(lic)
    upload_file(path_or_fileobj="/tmp/CPT_LICENSE", path_in_repo="LICENSE",
                repo_id=repo, revision=revision)
    log(f"LICENSE pushed to {repo}@{revision} ({len(lic):,} bytes)")


def _target_missing(model, targets):
    names = [n for n, _ in model.named_modules()]
    return [m for m in targets if not any(n == m or n.endswith("." + m) for n in names)]


# --------------------------------------------------------------------------- CPT
def run_cpt():
    # CCE (cut cross entropy) is unsupported for continued pre-training; must be
    # set before the trainer is built.
    os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
    cpt_tok = AutoTokenizer.from_pretrained(CFG["TOK_DIR"])
    assert cpt_tok.eos_token, "no eos_token; CPT would never learn to stop"
    eos = cpt_tok.eos_token

    dest = fetch(f'https://huggingface.co/datasets/{CFG["DATA_REPO"]}/resolve/main/{CFG["CPT_FILE"]}',
                 "/tmp/cpt_sample.jsonl")
    docs = [json.loads(l)["text"] for l in open(dest, encoding="utf-8") if l.strip()]
    cpt_ds = Dataset.from_dict({"text": [d + eos for d in docs]})
    log(f"CPT: {len(cpt_ds):,} documents")

    cpt_model, _ = FastLanguageModel.from_pretrained(
        model_name = CFG["BASE_REPO"], tokenizer_name = CFG["TOK_DIR"],
        max_seq_length = CFG["MAX_SEQ_LENGTH"], dtype = DTYPE,
        load_in_4bit = False, max_lora_rank = CFG["CPT_R"],
    )
    cpt_model = merge_if_adapter(cpt_model)

    # embed_tokens/lm_head are the mechanism for the new script (vendor: "add
    # embed_tokens and lm_head to allow the model to learn out of distribution
    # data"). LFM ties them, so Unsloth moves them to modules_to_save.
    cpt_targets = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3",
                   "embed_tokens", "lm_head"]
    missing = _target_missing(cpt_model, cpt_targets)
    if missing:
        raise RuntimeError(f"CPT target modules not present: {missing}")

    cpt_model = FastLanguageModel.get_peft_model(
        cpt_model, r = CFG["CPT_R"], lora_alpha = CFG["CPT_ALPHA"], lora_dropout = 0,
        bias = "none", target_modules = cpt_targets,
        use_gradient_checkpointing = "unsloth", random_state = 3407,
        use_rslora = True,   # True for CPT, False for SFT - the vendor differs deliberately
    )
    n = sum(p.numel() for p in cpt_model.parameters() if p.requires_grad)
    if n == 0:
        raise RuntimeError("CPT LoRA attached to zero parameters - check cpt_targets")
    log(f"CPT trainable {n/1e6:.1f}M")

    cpt_trainer = UnslothTrainer(
        model = cpt_model, tokenizer = cpt_tok,
        train_dataset = cpt_ds, eval_dataset = None,
        dataset_text_field = "text", max_seq_length = CFG["CPT_SEQ"],
        dataset_num_proc = 4,
        args = UnslothTrainingArguments(
            per_device_train_batch_size = CFG["CPT_BS"],
            gradient_accumulation_steps = CFG["CPT_ACC"],
            max_steps = CFG["CPT_STEPS"], warmup_steps = 10,
            learning_rate = CFG["CPT_LR"],
            embedding_learning_rate = CFG["CPT_EMB_LR"],
            logging_steps = max(5, CFG["CPT_STEPS"] // 60),
            optim = "adamw_8bit", weight_decay = 0.001, lr_scheduler_type = "linear",
            seed = 3407, output_dir = "cpt-out", save_strategy = "no", report_to = [],
        ),
    )
    cpt_trainer.add_callback(TimeCheckpoint(cpt_model, CFG["REPO"], CFG["CHECKPOINT_EVERY_S"],
                                            tokenizer = cpt_tok, revision = CFG["WIP"]))
    if IS_MAIN:
        log(f"CPT rows/step = {CFG['CPT_BS']} x {CFG['CPT_ACC']} x {WORLD_SIZE} GPU "
            f"= {CFG['CPT_BS'] * CFG['CPT_ACC'] * WORLD_SIZE}")
    cpt_stats = cpt_trainer.train()
    log(f"CPT metrics: {cpt_stats.metrics}")

    if IS_MAIN:
        cpt_model.generation_config.max_length = None
        cpt_model.generation_config.max_new_tokens = None
        # Publish the merged fp16 model AND the adapter, then hand the merged dir
        # to the SFT stage. This is the cpt-hindi branch the chain is built on.
        cpt_model.save_pretrained_merged(CFG["SFT_BASE"], tokenizer = cpt_tok,
                                         save_method = "merged_16bit")
        cpt_model.push_to_hub(CFG["REPO"], tokenizer = cpt_tok, revision = "cpt-hindi")
        from huggingface_hub import HfApi
        HfApi().upload_folder(repo_id = CFG["REPO"], revision = "cpt-hindi",
                              folder_path = CFG["SFT_BASE"],
                              ignore_patterns = ["*.pth", "cpt-out/*"])
        push_license(CFG["REPO"], "cpt-hindi")
        log(f"CPT done -> https://huggingface.co/{CFG['REPO']}/tree/cpt-hindi")

    # Let rank 0 finish writing the merged dir before the process group is torn
    # down; the SFT stage then loads it.
    if torch.distributed.is_initialized():
        torch.distributed.barrier()


# --------------------------------------------------------------------------- SFT
def run_sft():
    tok = AutoTokenizer.from_pretrained(CFG["TOK_DIR"])
    split = load_from_disk(CFG["SFT_SPLIT"])
    train_ds, eval_ds = split["train"], split["test"]
    log(f"SFT: train {len(train_ds):,} eval {len(eval_ds):,} rows")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = CFG["SFT_BASE"], tokenizer_name = CFG["TOK_DIR"],
        max_seq_length = CFG["MAX_SEQ_LENGTH"], dtype = DTYPE,
        load_in_4bit = False, max_lora_rank = 32,
    )
    model = merge_if_adapter(model)

    # LFM's MLP is w1/w2/w3 and its conv layers own in_proj/out_proj - not the
    # usual LLaMA names. A wrong list attaches nothing and still trains "fine".
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"]
    missing = _target_missing(model, target_modules)
    if missing:
        raise RuntimeError(f"Target modules not present in this model: {missing}")

    model = FastLanguageModel.get_peft_model(
        model, r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
        target_modules = target_modules, use_gradient_checkpointing = "unsloth",
        random_state = 3407, use_rslora = False,
    )
    # SFT must not overwrite the embeddings CPT just fitted on 24,000 documents.
    et = [n for n, p in model.named_parameters() if "embed_tokens" in n and p.requires_grad]
    if et:
        raise RuntimeError(f"SFT is training {et} - it must leave the CPT embeddings alone.")
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if n_train == 0:
        raise RuntimeError("LoRA attached to zero parameters - check target_modules")
    log(f"SFT trainable {n_train/1e6:.1f}M")

    trainer = UnslothTrainer(
        model = model, tokenizer = tokenizer,
        train_dataset = train_ds, eval_dataset = eval_ds,
        dataset_text_field = "text", max_seq_length = CFG["MAX_SEQ_LENGTH"],
        dataset_num_proc = 2, packing = False,
        args = UnslothTrainingArguments(
            per_device_train_batch_size = CFG["SFT_BS"],
            gradient_accumulation_steps = CFG["SFT_ACC"],
            max_steps = CFG["MAX_STEPS"], warmup_steps = 5,
            learning_rate = CFG["LEARNING_RATE"], lr_scheduler_type = "linear",
            optim = "adamw_8bit", weight_decay = 0.01,
            logging_steps = max(5, CFG["MAX_STEPS"] // 60),
            eval_strategy = "steps", eval_steps = 150,
            seed = 3407, output_dir = "hf-out", save_strategy = "no",
            group_by_length = True,
            report_to = ["wandb"] if CFG["WANDB_ON"] else [],
            run_name = CFG["BRANCH"],
        ),
    )
    # Mask the user turn so loss is computed on the ASSISTANT RESPONSE ONLY. Without
    # it the model is trained to emit the question as well as the answer.
    trainer = train_on_responses_only(
        trainer, instruction_part = "<|im_start|>user\n", response_part = "<|im_start|>assistant\n")
    _lab = trainer.train_dataset[0]["labels"]
    _masked = sum(1 for x in _lab if x == -100)
    if _masked == 0:
        raise RuntimeError("train_on_responses_only did not mask anything")
    log(f"label masking: {_masked}/{len(_lab)} tokens are -100")
    trainer.add_callback(TimeCheckpoint(model, CFG["REPO"], CFG["CHECKPOINT_EVERY_S"],
                                        tokenizer = tokenizer, revision = CFG["WIP"]))

    if IS_MAIN:
        log(f"SFT rows/step = {CFG['SFT_BS']} x {CFG['SFT_ACC']} x {WORLD_SIZE} GPU "
            f"= {CFG['SFT_BS'] * CFG['SFT_ACC'] * WORLD_SIZE}")

    torch.cuda.reset_peak_memory_stats()
    stats = trainer.train()
    log(f"peak {torch.cuda.max_memory_allocated()/2**30:.1f} GB "
        f"of {torch.cuda.get_device_properties(LOCAL_RANK).total_memory/2**30:.1f} GB")

    if IS_MAIN:
        print(str(stats.metrics), flush=True)
        print_eval(trainer)
        merge_gguf_and_push(model, tokenizer)

    if torch.distributed.is_initialized():
        torch.distributed.barrier()


def print_eval(trainer):
    """The eval-loss minimum is what decides the next run's step count, so print
    the curve and name the minimum instead of leaving it in a table."""
    hist = [(h.get("step"), h.get("loss"), h.get("eval_loss")) for h in trainer.state.log_history]
    ev = [(s, e) for s, l, e in hist if e is not None and s is not None]
    if not ev:
        return
    best = min(ev, key=lambda x: x[1])
    print(f"eval loss: {[(s, round(l, 4)) for s, l in ev]}", flush=True)
    print(f"  minimum at step {best[0]} = {best[1]:.4f}", flush=True)
    if ev[-1][1] > best[1]:
        print(f"  next run should stop near step {best[0]}, not {CFG['MAX_STEPS']}", flush=True)


def merge_gguf_and_push(model, tokenizer):
    import shutil
    from huggingface_hub import create_branch, model_info
    from huggingface_hub.utils import RevisionNotFoundError
    try:
        model_info(CFG["REPO"], revision=CFG["BRANCH"])
    except RevisionNotFoundError:
        log(f"branch '{CFG['BRANCH']}' not found - creating it")
        create_branch(CFG["REPO"], branch=CFG["BRANCH"], exist_ok=True)

    # Unsloth writes the f16 conversion to the cwd then moves it; on different
    # filesystems that is a 4.2 GB copy of a dir with little space, so chdir to
    # /tmp where the room is.
    shutil.rmtree("/kaggle/working/teacher", ignore_errors=True)
    os.chdir("/tmp")
    free = shutil.disk_usage("/tmp").free / 2**30
    print(f"cwd {os.getcwd()}  {free:.0f} GB free", flush=True)
    if free < 10:
        raise RuntimeError(f"only {free:.0f} GB on /tmp - not enough for the f16 pass")

    model.generation_config.max_length = None
    model.generation_config.max_new_tokens = None
    p = model.save_pretrained_gguf("hf-out", tokenizer, quantization_method = "q4_k_m")
    print("gguf at:", p, flush=True)
    model.push_to_hub_gguf(CFG["REPO"], tokenizer, quantization_method = "q4_k_m",
                           revision = CFG["BRANCH"])
    model.push_to_hub(CFG["REPO"], tokenizer, revision = CFG["BRANCH"])
    print(f"done -> https://huggingface.co/{CFG['REPO']}/tree/{CFG['BRANCH']}", flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", choices=["cpt", "sft"], required=True)
    a = ap.parse_args()
    log(f"stage={a.stage} world_size={WORLD_SIZE}")
    if a.stage == "cpt":
        run_cpt()
    else:
        run_sft()
    gc.collect()
    torch.cuda.empty_cache()


if __name__ == "__main__":
    main()

In [ ]:
import os, subprocess, sys   # cell-local: every cell must run after a kernel restart

SCRIPT = "/kaggle/working/train_ddp.py"

def _launch(stage, n_gpu=2):
    """One process per T4. torchrun sets LOCAL_RANK/WORLD_SIZE on each, which is
    what unsloth's planner reads to pin the model to that rank's own card."""
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone",
           f"--nproc_per_node={n_gpu}", SCRIPT, "--stage", stage]
    print("==>", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=os.environ.copy())

if RUN_CPT:
    _launch("cpt")        # continued pretraining on both T4s, merged to cpt-hindi
_launch("sft")            # SFT on both T4s, merged -> GGUF -> BRANCH
print("training done on 2 x T4; rank 0 pushed the merged GGUF + adapter")

## 5. Licence

The base is under **Liquid Open License v1.0** (`lfm1.0`), which has 16
redistribution clauses - the text must travel with the weights, not just the
model card name.

In [ ]:
# Liquid Open License v1.0 has 16 redistribution clauses. The text must travel with
# the weights, so it goes to this repo too.
#
# Self-contained on purpose: this cell is the one most likely to be re-run alone
# after a restart, and it died on a NameError for `os` that an earlier cell
# happened to have imported.
#
# The branch guard matters too. upload_file() raises RevisionNotFoundError when the
# target branch does not exist, and s2b-teacher was never created: the model push
# lives in the GGUF cell, which had not run. push_to_hub creates whatever branch
# IT is told to write, it does not create the others.
import os, urllib.request
from huggingface_hub import upload_file, create_branch, model_info
from huggingface_hub.utils import RevisionNotFoundError

REPO   = globals().get("REPO",   "kumarsujitroy/LFM2.5-1.2B-Instruct-HI-Uncensored")
BRANCH = globals().get("BRANCH", "s2b-teacher")

def ensure_branch(repo, rev):
    try:
        model_info(repo, revision=rev)
    except RevisionNotFoundError:
        print(f"branch '{rev}' not found - creating it")
        create_branch(repo, branch=rev, exist_ok=True)

ensure_branch(REPO, BRANCH)

_url = "https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct/resolve/main/LICENSE"
with urllib.request.urlopen(_url, timeout=180) as r:
    _lic = r.read()
os.makedirs("hf-out", exist_ok=True)
with open("hf-out/LICENSE", "wb") as f:
    f.write(_lic)
upload_file(path_or_fileobj="hf-out/LICENSE", path_in_repo="LICENSE",
            repo_id=REPO, revision=BRANCH)
print(f"LICENSE pushed to {REPO} @ {BRANCH}  ({len(_lic):,} bytes)")

## 6. Merge -> GGUF -> Hub (done by the training run)

The `train_ddp.py` launch ends on **rank 0** by merging the adapter, converting to
`q4_k_m` GGUF and pushing both to `BRANCH` - the work that used to be a separate cell.
The branch is the only thing that survives a Kaggle VM reset, so before ending the
session confirm it landed:

    https://huggingface.co/<you>/LFM2.5-1.2B-Instruct-HI-Uncensored/tree/s4-hinglish-v8

The five-minute `ckpt-*` adapters on `WIP` are the resume points if the session dies.

## 7. Back on the server

```
hf download <you>/lfm25-1.2b-bilingual-s2 --include "*Q4_K_M*" \
  --local-dir ~/finetune-models/s2          # NOT models/@models - see note
```

Always download to a per-session dir. `save_pretrained_gguf` names the file after
the **base** model, so s1/s2/s3 all produce `...Uncensored.Q4_K_M.gguf` and they
clobber each other in a shared directory.

Serve it and gate it:
```
nohup ~/llama.cpp/build/bin/llama-server \
  -m ~/finetune-models/s2/*Q4_K_M*.gguf -c 4096 --threads 2 \
  --host 127.0.0.1 --port 8081 --no-webui &
python3.13 /tmp/hindi.py 8081
```

**Gates, in priority order:**
1. `hindi 5/5` with **zero mojibake** - the whole point. Baseline before session 2
   is `0/5`, all romanized, zero mojibake: the model can *read* Devanagari but had
   never seen a Devanagari *reply*, so it answered in Latin script.
2. `hinglish 5/5` - must not regress
3. English intact
4. **Refusal compliance** - the abliteration lives in the weights you just
   finetuned; a language LoRA can wash it out. This is the likeliest failure.

Vendor decode settings (model card + docs.liquid.ai prompting guide), put these on
the llama-server line, not in chat.sh:

```
--temp 0.1 --top-k 50 --min-p 0.15 --repeat-penalty 1.05
```

chat.sh defaults to temp=1.0 and a jailbreak system prompt. Measured on this box:
greedy/t=0.5 collapses replies to ~5 words, t=1.0 rambles, and the vendor card sits
between them at 0.1. Sampling is polish, not the grammar fix; the grammar fix is v7.